Google Drive Link

In [ ]:
# Saves Results in Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = "/content/drive/MyDrive/melanoma/results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Required Libraries for Kaggle
os.system("pip install -q timm albumentations kaggle")

# Kaggle credentials (paste your own token below)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_################################"

print("SAVE_DIR:", SAVE_DIR, "| exists?", os.path.exists(SAVE_DIR))

Image Load

In [ ]:
import os
print("Images present?:", os.path.exists("/content/isic_data"))

In [ ]:
import os
if os.path.exists("/content/isic_data"):
    print("Data already present.")
else:
    os.system("kaggle datasets download -d sumaiyabinteshahid/isic-challenge-dataset-2020 -p /content/")
    os.system("unzip -q -o /content/isic-challenge-dataset-2020.zip -d /content/isic_data/")
    print("Download + unzip completed.")

In [ ]:
import glob, os, pandas as pd
CSV_PATH   = glob.glob("/content/isic_data/**/ISIC_2020_Train_Metadata.csv", recursive=True)[0]
IMAGES_DIR = os.path.dirname(glob.glob("/content/isic_data/**/train/ISIC_*.jpg", recursive=True)[0])
df = pd.read_csv(CSV_PATH)
print("CSV:", CSV_PATH, "\nIMG:", IMAGES_DIR)
print("columns:", list(df.columns))
print("img exists?:", os.path.exists(f"{IMAGES_DIR}/{df.iloc[0]['image_name']}.jpg"))

Functions and Classes

In [ ]:
import json, os, random, numpy as np, pandas as pd, cv2, torch, torch.nn as nn
import timm, albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG   = dict(backbone="efficientnet_b3", size=256, epochs=20, patience=7,
             batch=64, lr=3e-4, wd=1e-4, dropout=0.3, meta_hidden=64, workers=2)  # main models
STUDY = dict(backbone="efficientnet_b3", size=224, epochs=12, patience=5,
             batch=64, lr=3e-4, wd=1e-4, dropout=0.3, meta_hidden=64, workers=2)  # multi-seed evaluation
LOG = f"{SAVE_DIR}/multiseed_log.json"

def tfm(size, train):
    if train:
        return A.Compose([A.Resize(size, size), A.HorizontalFlip(p=.5), A.VerticalFlip(p=.5),
                          A.RandomRotate90(p=.5), A.Normalize(), ToTensorV2()])
    return A.Compose([A.Resize(size, size), A.Normalize(), ToTensorV2()])

class ISIC(Dataset):
    def __init__(self, d, train, size):
        self.d = d.reset_index(drop=True); self.t = tfm(size, train)
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        img = cv2.cvtColor(cv2.imread(f"{IMAGES_DIR}/{r['image_name']}.jpg"), cv2.COLOR_BGR2RGB)
        return self.t(image=img)["image"], torch.tensor(float(r["target"]))

class ISICMM(Dataset):
    def __init__(self, d, train, size, meta_cols):
        self.d = d.reset_index(drop=True); self.t = tfm(size, train); self.mc = meta_cols
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        img = cv2.cvtColor(cv2.imread(f"{IMAGES_DIR}/{r['image_name']}.jpg"), cv2.COLOR_BGR2RGB)
        img = self.t(image=img)["image"]
        meta = torch.tensor(r[self.mc].values.astype("float32"))
        return img, meta, torch.tensor(float(r["target"]))

class Net(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(cfg["backbone"], pretrained=True, num_classes=0)
        self.head = nn.Sequential(nn.Dropout(cfg["dropout"]), nn.Linear(self.backbone.num_features, 1))
    def forward(self, x): return self.head(self.backbone(x)).squeeze(1)

class MultiNet(nn.Module):
    def __init__(self, meta_dim, cfg):
        super().__init__()
        self.backbone = timm.create_model(cfg["backbone"], pretrained=True, num_classes=0)
        img_dim = self.backbone.num_features
        self.meta = nn.Sequential(nn.Linear(meta_dim, cfg["meta_hidden"]),
                                  nn.BatchNorm1d(cfg["meta_hidden"]), nn.ReLU(), nn.Dropout(cfg["dropout"]))
        self.head = nn.Sequential(nn.Dropout(cfg["dropout"]),
                                  nn.Linear(img_dim + cfg["meta_hidden"], 256), nn.ReLU(),
                                  nn.Dropout(cfg["dropout"]), nn.Linear(256, 1))
    def forward(self, x, m):
        return self.head(torch.cat([self.backbone(x), self.meta(m)], dim=1)).squeeze(1)

def prepare_data(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    d = pd.read_csv(CSV_PATH)
    sk = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)
    fo = list(sk.split(d, d["target"], d["patient_id"]))
    d["split"] = "train"
    d.iloc[fo[0][1], d.columns.get_loc("split")] = "test"
    d.iloc[fo[1][1], d.columns.get_loc("split")] = "val"
    trm = d["split"] == "train"
    d["sex_missing"] = d["sex"].isna().astype(int)
    d["sex_enc"] = d["sex"].map({"male": 1.0, "female": 0.0}).fillna(0.5)
    d["age_missing"] = d["age_approx"].isna().astype(int)
    d["age_filled"] = d["age_approx"].fillna(d.loc[trm, "age_approx"].median())
    d["age_z"] = (d["age_filled"] - d.loc[trm, "age_filled"].mean()) / (d.loc[trm, "age_filled"].std() + 1e-8)
    st = pd.get_dummies(d["anatom_site_general_challenge"].fillna("unknown"), prefix="site").astype(float)
    d = pd.concat([d, st], axis=1)
    MC = ["sex_enc", "sex_missing", "age_z", "age_missing"] + list(st.columns)
    return d, MC

def train_model(d, meta_cols, mm, cfg, save_prefix=None):
    def mk(split):
        ds = (ISICMM(d[d.split==split], split=="train", cfg["size"], meta_cols) if mm
              else ISIC(d[d.split==split], split=="train", cfg["size"]))
        return DataLoader(ds, batch_size=cfg["batch"], shuffle=(split=="train"),
                          num_workers=cfg["workers"], pin_memory=True, drop_last=(mm and split=="train"))
    ld = {s: mk(s) for s in ["train", "val", "test"]}
    model = (MultiNet(len(meta_cols), cfg) if mm else Net(cfg)).to(device)
    trm = d["split"] == "train"; pr = d[trm]["target"].mean()
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1 - pr) / pr], device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sc = torch.amp.GradScaler("cuda"); best, best_state, wait, hist = -1, None, 0, []
    def infer(loader):
        model.eval(); ys, ps = [], []
        with torch.no_grad():
            for batch in loader:
                if mm: x, m, y = batch; out = model(x.to(device), m.to(device))
                else:  x, y = batch;   out = model(x.to(device))
                ps.append(torch.sigmoid(out).float().cpu().numpy()); ys.append(y.numpy())
        return np.concatenate(ys), np.concatenate(ps)
    for ep in range(cfg["epochs"]):
        model.train()
        for batch in ld["train"]:
            if mm: x, m, y = batch; x, m, y = x.to(device), m.to(device), y.to(device)
            else:  x, y = batch;   x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast("cuda"):
                out = model(x, m) if mm else model(x); loss = crit(out, y)
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        vy, vp = infer(ld["val"]); va = roc_auc_score(vy, vp); hist.append(va)
        print(f"    epoch {ep:02d} | val AUC {va:.3f}")
        if va > best: best, best_state, wait = va, {k: v.cpu() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= cfg["patience"]: break
    model.load_state_dict(best_state)
    ty, tp = infer(ld["test"]); auc = float(roc_auc_score(ty, tp))
    if save_prefix:
        torch.save(best_state, f"{SAVE_DIR}/{save_prefix}_best.pt")
        np.save(f"{SAVE_DIR}/{save_prefix}_test_y.npy", ty)
        np.save(f"{SAVE_DIR}/{save_prefix}_test_p.npy", tp)
        np.save(f"{SAVE_DIR}/{save_prefix}_val_auc.npy", np.array(hist))
    return auc, ty, tp, np.array(hist)

def run_seed_study(seed):
    log = json.load(open(LOG)) if os.path.exists(LOG) else {}
    d, MC = prepare_data(seed)
    for kind, mm in [("baseline", False), ("multimodal", True)]:
        key = f"{kind}_{seed}"
        if key in log: print(f"skip {key} ({log[key]:.4f})"); continue
        print(f"--- {key} (size={STUDY['size']}, epochs={STUDY['epochs']}) ---")
        auc, _, _, _ = train_model(d, MC, mm, STUDY, save_prefix=None)
        log[key] = auc; json.dump(log, open(LOG, "w"))
        print(f"OK {key} | test AUC {auc:.4f} (saved)")
    print("Status:", log)

print("Setup ready. CFG:", CFG["size"], CFG["epochs"], "| STUDY:", STUDY["size"], STUDY["epochs"])

Model Run

In [ ]:
# Main models. If already trained (weights in Drive), skip and reuse them (no retraining).
import os
if os.path.exists(f"{SAVE_DIR}/baseline_best.pt") and os.path.exists(f"{SAVE_DIR}/multimodal_best.pt"):
    print("Trained models found in Drive - skipping training. Delete *_best.pt to retrain.")
else:
    d, MC = prepare_data(42)
    print("### BASELINE ###")
    b_auc, by, bp, b_hist = train_model(d, MC, mm=False, cfg=CFG, save_prefix="baseline")
    print(f"=== TEST baseline === AUC {b_auc:.4f}")
    print("### MULTIMODAL ###")
    m_auc, my, mp, m_hist = train_model(d, MC, mm=True, cfg=CFG, save_prefix="multimodal")
    print(f"Baseline {b_auc:.4f} -> Multimodal {m_auc:.4f} | Delta {m_auc-b_auc:+.4f}")


ROC-AUC Comparison

In [ ]:
import os
for f in ["baseline_test_y","baseline_test_p","multimodal_test_y","multimodal_test_p","baseline_val_auc","multimodal_val_auc"]:
    print("OK " if os.path.exists(f"{SAVE_DIR}/{f}.npy") else "MISSING", f)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

by = np.load(f"{SAVE_DIR}/baseline_test_y.npy");  bp = np.load(f"{SAVE_DIR}/baseline_test_p.npy")
my = np.load(f"{SAVE_DIR}/multimodal_test_y.npy"); mp = np.load(f"{SAVE_DIR}/multimodal_test_p.npy")
b_auc, m_auc = roc_auc_score(by, bp), roc_auc_score(my, mp)
b_hist = np.load(f"{SAVE_DIR}/baseline_val_auc.npy"); m_hist = np.load(f"{SAVE_DIR}/multimodal_val_auc.npy")
print(f"Baseline AUC {b_auc:.4f} | Multimodal AUC {m_auc:.4f} | Δ {m_auc-b_auc:+.4f}")

fig, ax = plt.subplots(2, 2, figsize=(12, 10))
for y, p, lab in [(by, bp, f"Image-only (AUC {b_auc:.3f})"), (my, mp, f"Multimodal (AUC {m_auc:.3f})")]:
    fpr, tpr, _ = roc_curve(y, p); ax[0,0].plot(fpr, tpr, lw=2, label=lab)
ax[0,0].plot([0,1],[0,1],"k--",lw=1); ax[0,0].set_title("ROC — test set")
ax[0,0].set_xlabel("1 - Specificity"); ax[0,0].set_ylabel("Sensitivity"); ax[0,0].legend(loc="lower right")

ax[0,1].bar(["Image-only","Multimodal"], [b_auc, m_auc], color=["#88aabb","#22aa66"])
ax[0,1].set_ylim(0.8, 0.92); ax[0,1].set_title(f"Test AUC (Δ = {m_auc-b_auc:+.3f})")
for i,v in enumerate([b_auc, m_auc]): ax[0,1].text(i, v+0.002, f"{v:.3f}", ha="center", fontweight="bold")

ax[1,0].plot(b_hist, "-o", label="Image-only", ms=4); ax[1,0].plot(m_hist, "-o", label="Multimodal", ms=4)
ax[1,0].set_title("Validation AUC per epoch"); ax[1,0].set_xlabel("epoch"); ax[1,0].set_ylabel("val AUC"); ax[1,0].legend()

cm = confusion_matrix(my, (mp>=0.5).astype(int))
ax[1,1].imshow(cm, cmap="Blues"); ax[1,1].set_title("Confusion matrix — multimodal")
ax[1,1].set_xticks([0,1]); ax[1,1].set_xticklabels(["benign","malignant"])
ax[1,1].set_yticks([0,1]); ax[1,1].set_yticklabels(["benign","malignant"])
ax[1,1].set_xlabel("predicted"); ax[1,1].set_ylabel("true")
for i in range(2):
    for j in range(2): ax[1,1].text(j,i,cm[i,j],ha="center",va="center",
                                     color="white" if cm[i,j]>cm.max()/2 else "black", fontweight="bold")
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/figures_comparison.png", dpi=150, bbox_inches="tight"); plt.show()
print("Saved:", f"{SAVE_DIR}/figures_comparison.png")

Multi-Seed Testing

In [ ]:
run_seed_study(42)

In [ ]:
run_seed_study(1)

In [ ]:
run_seed_study(7)

In [ ]:
import json, numpy as np
log = json.load(open(LOG))
b = np.array([log[f"baseline_{s}"]   for s in [42,1,7]])
m = np.array([log[f"multimodal_{s}"] for s in [42,1,7]])
print(f"Baseline    : {b.mean():.3f} ± {b.std():.3f}")
print(f"Multimodal : {m.mean():.3f} ± {m.std():.3f}")
print(f"Delta AUC   : {(m-b).mean():+.3f} ± {(m-b).std():.3f}")

Grad-CAM

In [ ]:
import glob, os
print("SAVE_DIR exists:", os.path.isdir(SAVE_DIR))
print(glob.glob(f"{SAVE_DIR}/*"))

In [ ]:
# =========================================================
#  GRAD-CAM CELL - interpretability of the baseline (image-only) model
#  Prerequisites: run the setup cells first; needs {SAVE_DIR}/baseline_best.pt
#  Inference ONLY: no re-training.
# =========================================================
import matplotlib.pyplot as plt
import torch.nn.functional as F

# --- 1) load trained baseline (size=256) ---
model = Net(CFG).to(device)
model.load_state_dict(torch.load(f"{SAVE_DIR}/baseline_best.pt", map_location=device))
model.eval()

# --- 2) layer target = last conv layer from EfficientNet-B3 ---
# (map space 8x8 to 256px)
target_layer = model.backbone.conv_head

# --- 3) hook: forward and backward ---
_feat, _grad = {}, {}
h1 = target_layer.register_forward_hook(lambda m, i, o: _feat.__setitem__("v", o.detach()))
h2 = target_layer.register_full_backward_hook(lambda m, gi, go: _grad.__setitem__("v", go[0].detach()))

# --- 4) Grad-CAM (single image) ---
def gradcam(img_tensor):
    model.zero_grad()
    x = img_tensor.unsqueeze(0).to(device)          # (1,3,H,W)
    score = model(x).squeeze()                      # scalar: logit "malignant"
    score.backward()                                # score gradient -> conv_head
    A = _feat["v"][0]                               # (C,h,w)  activations
    G = _grad["v"][0]                               # (C,h,w)  gradients
    w = G.mean(dim=(1, 2))                          # (C,)  weight = avg importance
    cam = torch.relu((w[:, None, None] * A).sum(0)) # (h,w) weighted sum + ReLU
    cam = cam / (cam.max() + 1e-8)                  # normalizing 0..1
    cam = F.interpolate(cam[None, None], size=(CFG["size"], CFG["size"]),
                        mode="bilinear", align_corners=False)[0, 0]
    return cam.cpu().numpy(), torch.sigmoid(score).item()

# --- 5) sample 4 malignant + 4 benign from the test set (seed 42) ---
d, _ = prepare_data(42)
test = d[d.split == "test"]
rows = pd.concat([test[test.target == 1].sample(4, random_state=42),
                  test[test.target == 0].sample(4, random_state=42)])
val_tfm = tfm(CFG["size"], train=False)

# --- 6) figure: row 1 = original, row 2 = Grad-CAM overlay ---
fig, ax = plt.subplots(2, 8, figsize=(24, 6.5))
for j, (_, r) in enumerate(rows.iterrows()):
    raw  = cv2.cvtColor(cv2.imread(f"{IMAGES_DIR}/{r['image_name']}.jpg"), cv2.COLOR_BGR2RGB)
    disp = cv2.resize(raw, (CFG["size"], CFG["size"]))
    cam, prob = gradcam(val_tfm(image=raw)["image"])
    lab = "MALIGNANT" if r.target == 1 else "benign"
    ax[0, j].imshow(disp); ax[0, j].axis("off")
    ax[0, j].set_title(f"{lab}\np={prob:.2f}",
                       color=("crimson" if r.target == 1 else "black"), fontsize=10)
    ax[1, j].imshow(disp); ax[1, j].imshow(cam, cmap="jet", alpha=0.45); ax[1, j].axis("off")
fig.text(0.5, 1.00, "Grad-CAM - where the model looks (red = high attention)",
         ha="center", fontsize=13)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/gradcam.png", dpi=130, bbox_inches="tight")
plt.show()

h1.remove(); h2.remove()   # clean hooks
print("Saved:", f"{SAVE_DIR}/gradcam.png")

In [ ]:
import numpy as np, torch
from sklearn.metrics import roc_auc_score, roc_curve

d, MC = prepare_data(42)
model = Net(CFG).to(device)
model.load_state_dict(torch.load(f"{SAVE_DIR}/baseline_best.pt", map_location=device))
model.eval()

def infer(split):
    ld = DataLoader(ISIC(d[d.split==split], False, CFG["size"]),
                    batch_size=CFG["batch"], num_workers=CFG["workers"], pin_memory=True)
    ys, ps = [], []
    with torch.no_grad():
        for x, y in ld:
            ps.append(torch.sigmoid(model(x.to(device))).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

vy, vp = infer("val")            # choose the threshold
ty, tp = infer("test")           # return metrics

fpr, tpr, thr = roc_curve(vy, vp)
thr_opt = thr[np.argmax(tpr - fpr)]              # Youden on validation
pred = (tp >= thr_opt).astype(int)
tn = ((pred==0)&(ty==0)).sum(); fp = ((pred==1)&(ty==0)).sum()
fn = ((pred==0)&(ty==1)).sum(); tp_= ((pred==1)&(ty==1)).sum()

print(f"AUC test            : {roc_auc_score(ty, tp):.3f}")
print(f"Threshold (Youden/val) : {thr_opt:.3f}")
print(f"Sensitivity test    : {tp_/(tp_+fn):.3f}")
print(f"Specificity test    : {tn/(tn+fp):.3f}")

Model Deployment

In [ ]:
import cv2, torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

THRESHOLD = 0.656   # threshold according to validation (Youden)

# resize + normalize, no augmentation
_infer_tfm = A.Compose([A.Resize(CFG["size"], CFG["size"]), A.Normalize(), ToTensorV2()])

# model load
model = Net(CFG).to(device)
model.load_state_dict(torch.load(f"{SAVE_DIR}/baseline_best.pt", map_location=device))
model.eval()   # <-- turn off dropout/batchnorm-training for inference

@torch.no_grad()                        # no gradients: faster, less memory
def predict_image(path, threshold=THRESHOLD):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    x = _infer_tfm(image=img)["image"].unsqueeze(0).to(device)   # (1,3,H,W)
    prob = torch.sigmoid(model(x)).item()                        # probability 0..1
    label = "MALIGNANT" if prob >= threshold else "benign"
    return label, prob

# usage:
label, prob = predict_image("/content/my_lesion.jpg")
print(f"Prediction: {label}  |  probability - malignant: {prob:.1%}  (threshold {THRESHOLD})")